## Create dataset with country-level monthly salary of nurses

 #### Load packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tjn_tools
import statsmodels.formula.api as smf

### Set paths,  define raw data & year
Data sources:
- Yearly salary of hospital nurses from OECD: https://stats.oecd.org/index.aspx?DataSetCode=HEALTH_REAC
- Average monthly earnings of employees by sex and education from ILO: https://www.ilo.org/ilostat-files/Documents/Excel/INDICATOR/EAR_4MTH_SEX_OCU_CUR_NB_A_EN.xlsx
- GDP and Population from World Bank: 
  - GDP: https://api.worldbank.org/v2/en/indicator/NY.GDP.MKTP.CD?downloadformat=csv, saved as "gdp_wb.csv"
  - Population: https://api.worldbank.org/v2/en/indicator/SP.POP.TOTL?downloadformat=csv, saved as "pop_wb_csv"

In [ ]:
# Set paths
path_raw = f"{tjn_tools.paths.source_data}/001 TJN/001d SOTJ"
path_final = f"{tjn_tools.paths.data_root}/Data requests/2023-05-03 Leo Hyde World PSI"

# Define raw data files
oecd_nurse_dataset = f"{path_raw}/HEALTH_REAC_03052023132902540.csv"
ilo_wages_dataset = f"{path_raw}/EAR_4MTH_SEX_OCU_CUR_NB_A_EN.xlsx"
wb_gdp_dataset = f"{path_raw}/gdp_wb.csv"
wb_population_dataset = f"{path_raw}/pop_wb.csv"
final_data = f"{path_final}/nurse_renumeration.csv"
# Define year (Note that for the OECD nurse renumeration, we take the latest available data as some of the countries are not updated so frequently)
year = 2020

### Create dataset with nurse salaries

Import and merge all relevant data

In [ ]:
# Import OECD data for renumeration of hospital nurses
oecd_nurses = pd.read_csv(oecd_nurse_dataset,
                         usecols=["Variable","Measure","COU","Year","Value"])
oecd_nurses = oecd_nurses.loc[(oecd_nurses["Variable"]=="Remuneration of hospital nurses") 
                              & (oecd_nurses["Measure"] == "Salaried, income, US$ exchange rate")]
oecd_nurses = oecd_nurses.sort_values('Year', ascending=False)
oecd_nurses = oecd_nurses.drop_duplicates(subset = ["COU"], keep='first')
oecd_nurses = oecd_nurses.loc[:,["COU","Value"]]
oecd_nurses.columns = ["iso3","wage_nurses_month"]
oecd_nurses["wage_nurses_month"] = oecd_nurses["wage_nurses_month"]/12

#Import World bank data
wb_gdp= pd.read_csv(wb_gdp_dataset, skiprows=range(4),
                         usecols=["Country Code", str(year)])
wb_gdp.columns = ["iso3","gdp_wb"]
wb_pop= pd.read_csv(wb_population_dataset, skiprows=range(4),
                         usecols=["Country Code", str(year)])
wb_pop.columns = ["iso3","pop_wb"]

# Import ILO data
ilo_wage = pd.read_excel(ilo_wages_dataset,
                         usecols=["Reference area","Sex","Occupation","Time","U.S. dollars"], skiprows=range(5))
ilo_wage = ilo_wage.loc[(ilo_wage["Occupation"]=="2. Professionals") & (ilo_wage["Time"]==year) & (ilo_wage["Sex"]=="Total")]
ilo_wage = ilo_wage.loc[:,["Reference area","U.S. dollars"]]
ilo_wage.loc[ilo_wage["Reference area"] == "Türkiye", "Reference area"] = "Turkey"
ilo_wage["iso3"] = ilo_wage["Reference area"].map(tjn_tools.get_iso3)
ilo_wage = ilo_wage.loc[:,["iso3","U.S. dollars"]]
ilo_wage.columns = ["iso3","wage_professionals_month"]

# Merge data
wages = pd.merge(oecd_nurses, ilo_wage, on='iso3', how = "outer")
wages = pd.merge(wages, wb_gdp, on='iso3', how = "outer")
wages = pd.merge(wages, wb_pop, on='iso3', how = "outer")

Complete dataset according to the following rules:
1. Take OECD nurse renumeration if it exists
2. Substitute missing data of nurse renumeration with ILO wages of professionals (which nurses belong to)
3. Impute missing professional wages as predicted by an OLS regressing professional wages on GDP/Capita (we impute professional, rather than nurse wages as the dataset on professional wages is bigger)

In [ ]:
# Take OECD nurse wages
wages["wage_nurses_month_filled"] = wages["wage_nurses_month"]
# replace missings with professional ILO wages
wages.loc[wages["wage_nurses_month_filled"].isna(),"wage_nurses_month_filled"] = wages.loc[wages["wage_nurses_month_filled"].isna(),"wage_professionals_month"]
# replace missings with imputed professional ILO wages
for i,x,y in zip(wages["iso3"],wages["gdp_wb"]/wages["pop_wb"],wages["wage_professionals_month"]):
    plt.annotate(i,(x,y))
plt.plot(wages["gdp_wb"]/wages["pop_wb"],wages["wage_professionals_month"],".")
plt.xscale("log")
plt.yscale("log")
# Delete outliers
wages.loc[wages["iso3"].isin(["BDI"]),"wage_professionals_month"] = np.nan
# Fill missing salaries
mod = smf.ols(formula='np.log(wage_professionals_month) ~ np.log(gdp_wb) + np.log(pop_wb)', data=wages)
res = mod.fit()
wages["wages_professionals_predicted"] = np.exp(res.predict(wages))
wages.loc[wages["wage_nurses_month_filled"].isna(),"wage_nurses_month_filled"] = wages.loc[wages["wage_nurses_month_filled"].isna(),"wages_professionals_predicted"]

Export dataset as .csv

In [ ]:
wages = wages.sort_values(by="iso3")
wages.to_csv(final_data, index=False)